# 문자열 알고리즘 개요

문자열 알고리즘은 문자열 간의 비교, 검색, 처리 등을 효율적으로 수행하기 위한 알고리즘입니다. 주로 다음과 같은 문제를 해결합니다:

- 특정 패턴이 본문 문자열에 존재하는지 확인
- 문자열에서 중복 부분 찾기
- 다수의 문자열을 빠르게 검색

## 주요 알고리즘 종류

| 알고리즘      | 특징 및 사용 목적                         |
|---------------|--------------------------------------------|
| KMP           | 실패함수를 이용한 빠른 문자열 매칭        |
| Rabin-Karp    | 해시 기반 빠른 매칭, 다중 패턴에 유리     |
| Boyer-Moore   | 뒤에서부터 비교하여 점프하는 최적화       |
| Trie          | 다수의 문자열 저장 및 검색에 적합         |
| Aho-Corasick  | Trie + Failure 링크로 다중 패턴 검색      |

---

# KMP 알고리즘 (Knuth-Morris-Pratt)

## 개요

KMP 알고리즘은 문자열 검색 알고리즘 중 하나로, 본문 문자열 내에서 **패턴 문자열을 O(N + M)** 시간 복잡도로 찾을 수 있습니다.

## 핵심 아이디어

- **중복된 비교를 줄이기 위해 실패함수(pi 배열)를 사용**
- 실패함수는 접두사와 접미사가 일치하는 최대 길이를 저장

---

## 1. 실패 함수 (pi 배열)

패턴 문자열의 각 위치에서 접두사 == 접미사의 최대 길이를 저장한 배열입니다.

예: 패턴 = `ababaca`  
→ pi = `[0, 0, 1, 2, 3, 0, 1]`

### 실패 함수 구하는 과정

In [ ]:
def compute_pi(pattern):
    m = len(pattern)
    pi = [0] * m
    j = 0  # 현재까지 일치한 접두사 길이

    for i in range(1, m):
        while j > 0 and pattern[i] != pattern[j]:
            j = pi[j - 1]
        if pattern[i] == pattern[j]:
            j += 1
            pi[i] = j
    return pi

In [1]:
import pandas as pd

In [2]:
pd.read_csv('./kmptb.csv')

,word 인덱스,word 문자,pattern 인덱스(pidx),pattern 문자,tb[pidx] 값,비교 결과,매칭 시작 인덱스
0,0,A,0,A,0,일치,NaN
1,1,B,1,B,0,일치,NaN
2,2,C,2,C,0,일치,NaN
3,3,,3,D,0,불일치 → pidx 이동,NaN
4,4,A,0,A,0,일치,NaN
5,5,B,1,B,0,일치,NaN
6,6,C,2,C,0,일치,NaN
7,7,D,3,D,0,일치,NaN
8,8,A,4,A,1,일치,NaN
9,9,B,5,B,2,일치,NaN


In [ ]:
import sys

def kmp_table(pattern):
    m = len(pattern)
    table = [0] * m
    j = 0

    for i in range(1, m):
        while j > 0 and pattern[i] != pattern[j]:
            j = table[j - 1]
        if pattern[i] == pattern[j]:
            j += 1
            table[i] = j
    return table

def kmp_search(text, pattern):
    n = len(text)
    m = len(pattern)
    table = kmp_table(pattern)
    result = []
    j = 0  # pattern index

    for i in range(n):  # text index
        while j > 0 and text[i] != pattern[j]:
            j = table[j - 1]
        if text[i] == pattern[j]:
            if j == m - 1:
                result.append(i - m + 2)  # 1-based index
                j = table[j]
            else:
                j += 1
    return result

# 입력 받기 (백준 스타일)
def main():
    T = sys.stdin.readline().rstrip()
    P = sys.stdin.readline().rstrip()

    positions = kmp_search(T, P)
    print(len(positions))
    print(' '.join(map(str, positions)))

if __name__ == "__main__":
    main()

    
# 입력
# ABC ABCDAB ABCDABCDABDE
# ABCDABD

# 출력
# 1
# 16

In [ ]:
import sys

# 1. 실패 테이블 생성 함수
def kmp_table(pattern):
    m = len(pattern)
    table = [0] * m  # 실패 테이블 초기화
    j = 0  # 접두사 길이 (pattern[0 ~ j-1] 일치 길이)

    # 패턴의 1번 인덱스부터 끝까지 순회
    for i in range(1, m):
        # 일치하지 않을 경우 → 접두사 길이 j를 줄여가며 비교
        while j > 0 and pattern[i] != pattern[j]:
            j = table[j - 1]

        # 일치하면 j 증가하고 해당 위치의 테이블 값을 갱신
        if pattern[i] == pattern[j]:
            j += 1
            table[i] = j

    return table  # 완성된 실패 테이블 반환

# 2. KMP 검색 함수
def kmp_search(text, pattern):
    n = len(text)
    m = len(pattern)
    table = kmp_table(pattern)  # 실패 테이블 미리 계산
    result = []  # 패턴이 일치한 위치를 저장할 리스트
    j = 0  # 패턴 인덱스

    # 텍스트 전체를 순회
    for i in range(n):
        # 불일치 시, 실패 테이블을 참고해 j를 뒤로 이동
        while j > 0 and text[i] != pattern[j]:
            j = table[j - 1]

        # 현재 문자가 일치하면
        if text[i] == pattern[j]:
            # 패턴 끝까지 도달하면 일치 성공
            if j == m - 1:
                result.append(i - m + 2)  # 위치 기록 (1-based index)
                j = table[j]  # 다음 탐색을 위해 j 초기화
            else:
                j += 1  # 다음 문자 비교를 위해 j 증가

    return result  # 일치 위치 리스트 반환

# 3. 입력 처리 및 출력
def main():
    # 표준 입력으로 문자열 읽기 (줄바꿈 제거)
    T = sys.stdin.readline().rstrip()
    P = sys.stdin.readline().rstrip()

    # KMP 탐색 실행
    positions = kmp_search(T, P)

    # 결과 출력
    print(len(positions))  # 패턴이 일치한 횟수
    print(' '.join(map(str, positions)))  # 각 일치 위치 출력 (1-based index)

# 4. 실행 부분
if __name__ == "__main__":
    main()


### 💡 예제 문제

문제 설명
문자열 T = "aabaabaafa" 와 패턴 P = "aabaa" 가 주어졌을 때,
KMP 알고리즘을 사용하여 패턴이 텍스트에 등장하는 위치를 모두 출력하시오.

입력

T: aabaabaafa

P: aabaa

출력

1

4